# 🚀 QuantDataPipeline — 量化數據中台

**全自動一鍵執行**：輸入參數 → 按下播放鍵 → 自動下載 + Greeks 計算 + 多週期特徵聚合

---

### 📋 使用說明
1. 在下方表單填入 **FinMind API Token**
2. 設定 **每小時 API 額度** (帳號等級對應的 requests/hour)
3. 選擇 **GitHub 分支**、**回溯天數** (填 `0` = 全量抓取到 2011-01-03)
4. 按下左側 ▶️ 播放鍵即可

> ⚠️ 首次執行會自動安裝依賴套件（約 30 秒），後續執行會直接跳過。
>
> ⚠️ 2019-01-16 ~ 2019-06-30 期間部分資料不完整 (FinMind 官方已知缺失)。

In [ ]:

#@title 🎛️ QuantDataPipeline 控制面板 { run: "auto", display-mode: "form" }
#@markdown ---
#@markdown ### 🔑 API 設定
FINMIND_API_TOKEN = '' #@param {type:"string"}
#@markdown ---
#@markdown ### 🌿 GitHub 分支
BRANCH = '6' #@param {type:"string"}
#@markdown ---
#@markdown ### ⚙️ 管線參數
LOOKBACK_DAYS = 0 #@param {type:"integer"}
#@markdown > `0` = 全量抓取 (從 2011-01-03 至今)。資料區間: 2011-01-03 ~ now
SKIP_GREEKS = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### ⚡ V2 並行加速設定
DOWNLOAD_WORKERS = 30 #@param {type:"integer"}
#@markdown > 下載 API 同時併發數 (建議 30~60)
GREEKS_WORKERS = 0 #@param {type:"integer"}
#@markdown > Greeks 計算核心數 (`0` = 自動偵測 Colab 所有核心)
#@markdown ---
#@markdown ### 💾 儲存與 Drive 同步
SYNC_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_PATH = '/content/drive/MyDrive/QuantData' #@param {type:"string"}
RESTORE_FROM_DRIVE = False #@param {type:"boolean"}
#@markdown > 
#@markdown > **何時打勾 (斷線續傳救援)**：當 Colab 斷線重開，且 status.db 顯示已下載，
#@markdown > 但本地沒有 Parquet 導致無法算 Greeks 時，打勾讓它從 Drive 抓回檔案補算。
#@markdown > **平常不需打勾**，維持 False 速度最快。
CLEANUP_AFTER_SYNC = True #@param {type:"boolean"}
#@markdown > 每月處理完且同步到 Drive 後，自動刪除本地 Parquet，節省大把 Colab 空間 ♻️

# ═══════════════════════════════════════════════════════════════
# 以下為自動執行邏輯，不需修改 (V2架構 - 月批次處理)
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os, time, shutil, json, logging, math
from datetime import datetime, timedelta, timezone
from pathlib import Path
from IPython.display import display, HTML, clear_output
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import multiprocessing

# ── 設定台北時區 ──
TZ_TPE = timezone(timedelta(hours=8))

def log(icon, msg):
    ts = datetime.now(TZ_TPE).strftime('%H:%M:%S')
    print(f"{icon} {ts} | {msg}", flush=True)

def header(title):
    print(f"\n{'━'*60}", flush=True)
    print(f"  {title}", flush=True)
    print(f"{'━'*60}", flush=True)



# ── 清除舊模組快取 ──
stale_prefixes = ['core.', 'fetchers.', 'processors.', 'storage.', 'compute_greeks']
for mod_name in list(sys.modules.keys()):
    if any(mod_name.startswith(p) or mod_name == p.rstrip('.') for p in stale_prefixes):
        del sys.modules[mod_name]

# 取消 RateLimiter 警告 (V2 我們直接多線程衝API，如果觸發再退避)
for name in ['pipeline', 'pipeline.retry', 'pipeline.extractor', 'pipeline.orchestrator', 'pipeline.http', 'pipeline.rate_limiter', 'pipeline.schema']:
    logging.getLogger(name).setLevel(logging.CRITICAL)

os.environ['FINMIND_API_TOKEN'] = FINMIND_API_TOKEN

# CSS + JS 優化顯示框
display(HTML("""
<style>
  .output_scroll { height: 450px !important; overflow-y: auto !important; }
  .output_wrapper { max-height: 450px !important; overflow-y: auto !important; }
  .output_area pre { font-family: 'Fira Code', 'Consolas', monospace; font-size: 13px; line-height: 1.5; }
</style>
<script>
  (function() {
    var output = document.querySelector('.output_scroll, .output_wrapper');
    if (output) { output.style.maxHeight = '450px'; output.style.overflowY = 'auto'; }
    var observer = new MutationObserver(function() {
      var el = document.querySelector('.output_scroll, .output_wrapper');
      if (el) el.scrollTop = el.scrollHeight;
    });
    var target = document.querySelector('.output_area');
    if (target) observer.observe(target, {childList: true, subtree: true});
  })();
</script>
"""))

pipeline_start_time = time.time()

# ── 日期計算 ──
DATA_EARLIEST = '2011-01-03'
today = datetime.now(TZ_TPE)
end_date = today.strftime('%Y-%m-%d')
if LOOKBACK_DAYS <= 0:
    start_date = DATA_EARLIEST
    lookback_label = f'全量 ({DATA_EARLIEST} ~ {end_date})'
else:
    start_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    lookback_label = f'{LOOKBACK_DAYS} 天 ({start_date} ~ {end_date})'

header('🚀 QuantDataPipeline (V2 批次架構) 啟動中')
log('📅', f'範圍: {lookback_label}')
log('🔧', f'Greeks: {"開" if not SKIP_GREEKS else "關"} | 分支: {BRANCH}')
log('⚡', f'並行: 下載 {DOWNLOAD_WORKERS} workers | Greeks {GREEKS_WORKERS if GREEKS_WORKERS > 0 else "自動"} workers')
log('💾', f'本地清理: {"開啟 (強力推薦)" if CLEANUP_AFTER_SYNC else "關閉"} | 網路恢復: {"開" if RESTORE_FROM_DRIVE else "關"}')
print(flush=True)

# ═══════════════════════════════════════════════════════════════
# Phase 0: 環境準備
# ═══════════════════════════════════════════════════════════════
header('📦 Phase 0: 環境準備')

REPO_URL = 'https://github.com/hsp1234-web/SP_OP_20260220.git'
REPO_DIR = Path('/content/SP_OP_20260220')
drive_data = Path(DRIVE_PATH) if SYNC_TO_DRIVE else None

if SYNC_TO_DRIVE:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        log('✅', 'Google Drive 已掛載')
    except Exception as e:
        log('⚠️', f'Drive 掛載失敗，關閉同步。錯誤: {str(e)[:40]}')
        SYNC_TO_DRIVE = False
        drive_data = None

if REPO_DIR.exists():
    log('🔄', f'更新代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=str(REPO_DIR), capture_output=True)
else:
    log('📥', f'下載代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True)

PROJECT_DIR = REPO_DIR / 'QuantDataPipeline'
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

log('📦', '安裝 Python 依賴包 (含 tqdm)...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'polars', 'numba', 'scipy', 'numpy', 'requests', 'python-dotenv', 'tqdm'],
    capture_output=True, text=True
)
log('✅', '依賴套件就緒')
os.chdir(str(PROJECT_DIR))

# 寫入 .env
env_path = PROJECT_DIR / '.env'
env_path.write_text(f'FINMIND_API_TOKEN={FINMIND_API_TOKEN}\nFINMIND_QUOTA_PER_HOUR=1600\n')
os.environ['FINMIND_API_TOKEN'] = FINMIND_API_TOKEN

# 匯入專案模組
import requests
from core.config import DATA_DIR, DB_PATH
from core.db_metadata_manager import DBManager
from fetchers.datasets.technical import trading_date
from fetchers.parsers.finmind_extractor import extract_and_cast
from storage.parquet_writer import save_dataframe
if not SKIP_GREEKS:
    from compute_greeks_pipeline import compute_greeks_for_date

# 還原 status.db
if SYNC_TO_DRIVE and drive_data:
    drive_db = drive_data / 'status.db'
    if drive_db.exists():
        shutil.copy2(drive_db, DB_PATH)
        log('📋', '已從 Drive 還原狀態庫 (status.db)')

DBManager._reset_instance()
db = DBManager(DB_PATH)

# 確認交易日
try:
    log('🔍', '獲取交易日曆...')
    from fetchers.infrastructure.http_session import get_session
    session = get_session()
    t_df = trading_date.fetch_trading_dates(session, start_date, end_date)
    if t_df is not None and not t_df.is_empty():
        d_col = 'date' if 'date' in t_df.columns else t_df.columns[0]
        all_dates = sorted(t_df[d_col].cast(str).to_list(), reverse=True)
        all_dates = [d[:10] for d in all_dates]
        log('✅', f'找到 {len(all_dates)} 個交易日')
    else:
        log('⚠️', '此範圍內無交易日')
        all_dates = []
except Exception as e:
    log('❌', f'取得交易日失敗 (請確認 API Token 有效): {str(e)[:60]}')
    all_dates = []

if not all_dates:
    sys.exit(0)

# ═══════════════════════════════════════════════════════════════
# V2 演算法：按月分批 (YYYY-MM)
# ═══════════════════════════════════════════════════════════════

from collections import defaultdict
months_map = defaultdict(list)
for d in all_dates:
    month = d[:7]  # 'YYYY-MM'
    months_map[month].append(d)

monthly_batches = sorted(months_map.items(), key=lambda x: x[0], reverse=True)

# ── 工作函式與Drive同步函式 ──
def download_worker(task):
    date, dataset, data_id = task
    url = "https://api.finmindtrade.com/api/v4/data"
    params = {"dataset": dataset, "data_id": data_id, "start_date": date, "end_date": date}
    if FINMIND_API_TOKEN: params["token"] = FINMIND_API_TOKEN
        
    retries = 3
    for attempt in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            if resp.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            
            resp.raise_for_status()
            res_json = resp.json()
            data_count = len(res_json.get("data", []))
            
            if data_count == 0: return (task, 3, None)
            
            df = extract_and_cast(res_json, dataset)
            if df is not None and not df.is_empty():
                path, checksum = save_dataframe(df, dataset, date, data_id)
                if checksum: return (task, 1, checksum)
            return (task, 3, None)
        except Exception as e:
            if "permission" in str(e).lower() or "level" in str(e).lower(): return (task, -2, "Permission Denied. 權限不足")
            if attempt == retries - 1: return (task, -1, str(e))
    return (task, -1, "Retry exhausted")

def restore_from_drive(f_path: Path):
    if not SYNC_TO_DRIVE or not drive_data: return False
    rel = f_path.relative_to(DATA_DIR)
    d_path = drive_data / 'data' / rel
    if d_path.exists():
        f_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(d_path, f_path)
        return True
    return False

def sync_and_verify(month_str, dates):
    if not SYNC_TO_DRIVE or not drive_data: return 0, 0
    synced = 0; passed = 0
    for d in dates:
        y_str = d[:4]
        txo_p = DATA_DIR / y_str / 'TaiwanOptionTick' / f'TXO_{d}.parquet'
        tx_p  = DATA_DIR / y_str / 'TaiwanFuturesTick' / f'TX_{d}.parquet'
        gk_p  = DATA_DIR / y_str / 'GreeksFeatures' / f'TXO_Greeks_{d}.parquet'
        txo_ok, tx_ok, gk_ok = False, False, False
        
        if txo_p.exists():
            dest = drive_data / 'data' / y_str / 'TaiwanOptionTick' / f'TXO_{d}.parquet'
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or dest.stat().st_size != txo_p.stat().st_size:
                shutil.copy2(txo_p, dest); synced += 1
            txo_ok = True
        elif (drive_data / 'data' / y_str / 'TaiwanOptionTick' / f'TXO_{d}.parquet').exists(): txo_ok = True
            
        if tx_p.exists():
            dest = drive_data / 'data' / y_str / 'TaiwanFuturesTick' / f'TX_{d}.parquet'
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or dest.stat().st_size != tx_p.stat().st_size:
                shutil.copy2(tx_p, dest); synced += 1
            tx_ok = True
        elif (drive_data / 'data' / y_str / 'TaiwanFuturesTick' / f'TX_{d}.parquet').exists(): tx_ok = True
            
        if not SKIP_GREEKS:
            if gk_p.exists():
                dest = drive_data / 'data' / y_str / 'GreeksFeatures' / f'TXO_Greeks_{d}.parquet'
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists() or dest.stat().st_size != gk_p.stat().st_size:
                    shutil.copy2(gk_p, dest); synced += 1
                gk_ok = True
            elif (drive_data / 'data' / y_str / 'GreeksFeatures' / f'TXO_Greeks_{d}.parquet').exists(): gk_ok = True
        else: gk_ok = True
            
        db_tid = f"{d}_TaiwanOptionTick_TXO"
        if txo_ok and tx_ok and gk_ok:
            db.update_task_status(db_tid, 2); passed += 1
        elif txo_ok or tx_ok:
            if db.get_task_status(db_tid) != 3: db.update_task_status(db_tid, 1)
    
    if synced > 0 or passed > 0:
        shutil.copy2(DB_PATH, drive_data / 'status.db')
    return synced, passed

def cleanup_local_month(dates):
    if not CLEANUP_AFTER_SYNC: return 0
    deleted_mb = 0
    for d in dates:
        for sub in ['TaiwanOptionTick', 'TaiwanFuturesTick', 'GreeksFeatures']:
            pref = "TX_" if sub == "TaiwanFuturesTick" else "TXO_Greeks_" if sub == "GreeksFeatures" else "TXO_"
            p = DATA_DIR / d[:4] / sub / f"{pref}{d}.parquet"
            if p.exists():
                deleted_mb += p.stat().st_size / (1024*1024)
                p.unlink()
    return deleted_mb

def emergency_backup():
    print("\n\n") # 換行避免蓋到進度條
    log('⚠️', '收到停止指令 (KeyboardInterrupt)！準備安全關機...')
    if SYNC_TO_DRIVE and drive_data:
        try:
            drive_db = drive_data / 'status.db'
            shutil.copy2(DB_PATH, drive_db)
            log('💾', '緊急救援：已將最新任務進度 (status.db) 安全備份至 Google Drive')
            log('🏁', '程式已安全中斷。下次重跑將從斷點自動繼續。')
        except Exception as e:
            log('⛔', f'緊急備份進度失敗: {e}')
    else:
        log('🏁', '程式已中斷。(狀態目前保留於本地 status.db)')

# ── 執行主循環 ──
total_months = len(monthly_batches)
workers_dl = DOWNLOAD_WORKERS
workers_gk = GREEKS_WORKERS if GREEKS_WORKERS > 0 else (multiprocessing.cpu_count() or 2)

total_api, total_gk, total_sync = 0, 0, 0
greeks_failed_dates = []

header('⚡ V2 月批次處理開始 ( 下載 🔀 Greeks 🔀 同步/清理 )')

try:
    for m_idx, (month, dates) in enumerate(monthly_batches, 1):
        m_label = f"[{m_idx}/{total_months}] {month} ({len(dates)}天)"
        log('🗓️', f'==== 開始處理: {m_label} ====')
        
        tasks_to_download = []
        need_greeks = []
        
        for d in dates:
            st_txo = db.get_task_status(f"{d}_TaiwanOptionTick_TXO")
            st_tx  = db.get_task_status(f"{d}_TaiwanFuturesTick_TX")
            if (st_txo == 2) or (st_txo == 3 and st_tx == 3): continue
                
            txo_p = DATA_DIR / d[:4] / 'TaiwanOptionTick' / f'TXO_{d}.parquet'
            if not txo_p.exists():
                if RESTORE_FROM_DRIVE and restore_from_drive(txo_p): pass
                elif st_txo != 3: tasks_to_download.append((d, 'TaiwanOptionTick', 'TXO'))
                    
            tx_p = DATA_DIR / d[:4] / 'TaiwanFuturesTick' / f'TX_{d}.parquet'
            if not tx_p.exists():
                if RESTORE_FROM_DRIVE and restore_from_drive(tx_p): pass
                elif st_tx != 3: tasks_to_download.append((d, 'TaiwanFuturesTick', 'TX'))
                    
            need_greeks.append(d)

        # ── 下載階段 ──
        if tasks_to_download:
            tot = len(tasks_to_download)
            log('  ', f'📥 準備下載 {tot} 任務 (進度條於下方更新)')
            t0 = time.time()
            success, skips, errs = 0, 0, 0
            fatal_error = False
            done_count = 0
            
            with ThreadPoolExecutor(max_workers=workers_dl) as pool:
                futures = {pool.submit(download_worker, t): t for t in tasks_to_download}
                try:
                    pbar = tqdm(as_completed(futures), total=tot, desc="📥 下載進度", leave=False, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} {postfix}")
                    for fut in pbar:
                        t, st, res = fut.result()
                        db_tid = f"{t[0]}_{t[1]}_{t[2]}"
                        if st == 1:
                            db.update_task_status(db_tid, 1); success += 1
                        elif st == 3:
                            db.update_task_status(db_tid, 3); skips += 1
                        elif st == -2:
                            fatal_error = True
                        else:
                            db.update_task_status(db_tid, 0); errs += 1
                        
                        err_str = f" ❌{errs}" if errs > 0 else ""
                        pbar.set_postfix_str(f"({success}✅ {skips}➖{err_str})")
                    pbar.close()
                except KeyboardInterrupt:
                    pool.shutdown(wait=False, cancel_futures=True)
                    raise
            if fatal_error:
                log('⛔', '遇到嚴重權限錯誤，退出管線。請確認 API Token 有 backer 以上權限。')
                sys.exit(1)
                
            total_api += tot
            log('  ', f'✅ 下載完畢 ({time.time()-t0:.1f}s) | 成功:{success}, 空:{skips}, 失敗:{errs}')
        else:
             log('  ', '📥 本月無須下載資料 (皆已存在或完成)')

        # ── 計算階段 ──
        if need_greeks and not SKIP_GREEKS:
            calc_targets = []
            for d in need_greeks:
                txo_p = DATA_DIR / d[:4] / 'TaiwanOptionTick' / f'TXO_{d}.parquet'
                tx_p  = DATA_DIR / d[:4] / 'TaiwanFuturesTick' / f'TX_{d}.parquet'
                gk_p  = DATA_DIR / d[:4] / 'GreeksFeatures' / f'TXO_Greeks_{d}.parquet'
                if gk_p.exists(): continue
                if not txo_p.exists() or not tx_p.exists():
                    if db.get_task_status(f"{d}_TaiwanOptionTick_TXO") not in (3, 2):
                        db.update_task_status(f"{d}_TaiwanOptionTick_TXO", 0)
                    continue
                calc_targets.append(d)
            
            if calc_targets:
                tot = len(calc_targets)
                log('  ', f'🧮 準備計算 Greeks: {tot} 天 (進度條於下方更新)')
                t1 = time.time()
                gk_done, gk_err = 0, 0
                done_count = 0
                
                with ProcessPoolExecutor(max_workers=workers_gk) as ppool:
                    gk_futures = {ppool.submit(compute_greeks_for_date, d): d for d in calc_targets}
                    try:
                        pbar = tqdm(as_completed(gk_futures), total=tot, desc="🧮 計算進度", leave=False, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} {postfix}")
                        for fut in pbar:
                            d = gk_futures[fut]
                            try:
                                df, path, ok = fut.result()
                                if ok: gk_done += 1
                                else: gk_err += 1; greeks_failed_dates.append(d)
                            except KeyboardInterrupt:
                                raise
                            except Exception as e:
                                gk_err += 1; greeks_failed_dates.append(d)
                                pbar.write(f"    ❌ {d} Greeks Error: {str(e)[:40]}")
                                
                            err_str = f" ❌{gk_err}" if gk_err > 0 else ""
                            pbar.set_postfix_str(f"({gk_done}✅{err_str})")
                        pbar.close()
                    except KeyboardInterrupt:
                        # 不讓 ProcessPoolExecutor 產生掛死
                        for f in gk_futures:
                            f.cancel()
                        ppool.shutdown(wait=False)
                        raise
                total_gk += gk_done
                log('  ', f'✅ 計算完畢 ({time.time()-t1:.1f}s) | 完成:{gk_done}, 失敗:{gk_err}')
            else:
                log('  ', '🧮 本月無須重複計算 Greeks')

        # ── 同步階段 ──
        if SYNC_TO_DRIVE:
            s_cnt, p_cnt = sync_and_verify(month, dates)
            total_sync += s_cnt
            if s_cnt > 0 or p_cnt > 0:
                log('  ', f'☁️ 同步與驗證: 備份 {s_cnt} 個檔 | 驗證通過 {p_cnt} 天 🏆')
            else:
                log('  ', '☁️ 同步與驗證: 備份 0 個檔 | 無新增驗證')

        # ── 清理階段 ──
        if CLEANUP_AFTER_SYNC:
            freed = cleanup_local_month(dates)
            if freed > 0:
                log('  ', f'♻️ 本地清理: 釋放 {freed:.1f} MB 空間')

    # ═══════════════════════════════════════════════════════════════
    # 完成統計
    # ═══════════════════════════════════════════════════════════════
    elapsed_min = round((time.time() - pipeline_start_time) / 60, 1)

    header('🏁 管線執行完畢 (V2 架構)')
    log('📊', f'總耗時:     ~{elapsed_min} 分鐘')
    log('  ', f'API 呼叫:   {total_api} 次')
    if not SKIP_GREEKS: log('  ', f'Greeks 產出: {total_gk} 天')
    if SYNC_TO_DRIVE: log('  ', f'Drive 備份: {total_sync} 個檔案')
    if CLEANUP_AFTER_SYNC: log('  ', 'Colab 本地維持輕量狀態 ✅')

    if greeks_failed_dates:
        log('⚠️', f'有 {len(greeks_failed_dates)} 天的 Greeks 計算失敗，待日後排查:')
        err_list = sorted(list(set(greeks_failed_dates)), reverse=True)
        log('  ', f'{", ".join(err_list[:10])}' + (' ...' if len(err_list)>10 else ''))

except KeyboardInterrupt:
    emergency_backup()

